# ASVspoof5 Train/Dev Subset Preparation (A01-A16, Globally Balanced)

This notebook builds two speaker-disjoint manifests for later logistic-regression training:

- `train` source partition: `bonafide`, `A01` ... `A08`
- `dev` source partition: `bonafide`, `A09` ... `A16`

The objective in this version is:
- use the full eligible speaker pool in each source partition
- split speakers internally into `train/test`
- choose speaker splits that support the largest balanced class counts
- enforce one **global common quota** so every class from `A01` to `A16` and `bonafide` has the same number of samples per split

That means if the weakest class/split can support `N_train` and `N_test`, every task will use exactly:
- `N_train` bonafide + `N_train` spoof for training
- `N_test` bonafide + `N_test` spoof for testing

Outputs per source partition:
- `selected_speakers.csv`
- `selected_utterances_plan.csv`
- `selection_audit.csv`
- `plan_summary.json`


In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [4]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
PROTOCOL_DIR = PROJECT_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_protocols'

PARTITION_SPECS = {
    'train': {
        'protocol_path': PROTOCOL_DIR / 'ASVspoof5.train.tsv',
        'target_systems': [f'A{i:02d}' for i in range(1, 9)],
    },
    'dev': {
        'protocol_path': PROTOCOL_DIR / 'ASVspoof5.dev.track_1.tsv',
        'target_systems': [f'A{i:02d}' for i in range(9, 17)],
    },
}

OUT_BASE = PROTOCOL_DIR / 'train_dev_16_systems_outputs'
OUT_BASE.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_SPEAKER_RATIO = 0.7
MIN_TEST_SPEAKERS = 2
MAX_SELECTION_TRIES = 2000

COLS = ['speaker_id','utt_id','gender','c3','c4','c5','codec_id','system_id','label','unused']

print('OUT_BASE =', OUT_BASE)
for part, spec in PARTITION_SPECS.items():
    print(part, 'protocol =', spec['protocol_path'], '| exists =', spec['protocol_path'].exists())


OUT_BASE = /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs
train protocol = /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/ASVspoof5.train.tsv | exists = True
dev protocol = /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/ASVspoof5.dev.track_1.tsv | exists = True


In [5]:
# ===== Load protocols =====
def load_partition_df(protocol_path: Path, target_systems: list[str], source_partition: str) -> pd.DataFrame:
    assert protocol_path.exists(), f'Missing protocol: {protocol_path}'
    df = pd.read_csv(protocol_path, sep=r'\s+', header=None, names=COLS, engine='python')
    df['label'] = df['label'].astype(str).str.strip().replace({'genuine': 'bonafide'})
    df['system_id'] = df['system_id'].astype(str).str.strip()
    df.loc[df['label'].eq('bonafide') & ~df['system_id'].eq('bonafide'), 'system_id'] = 'bonafide'
    df['sample_class'] = np.where(df['label'].eq('bonafide'), 'bonafide', df['system_id'])
    keep = {'bonafide', *target_systems}
    df = df[df['sample_class'].isin(keep)].copy().reset_index(drop=True)
    df['source_partition'] = source_partition
    return df

partition_frames = {
    part: load_partition_df(spec['protocol_path'], spec['target_systems'], part)
    for part, spec in PARTITION_SPECS.items()
}

for part, df in partition_frames.items():
    print(f'\n{part.upper()} rows =', len(df))
    print('speakers =', df['speaker_id'].nunique())
    print(df['sample_class'].value_counts().sort_index())



TRAIN rows = 182357
speakers = 400
A01         20445
A02         20445
A03         20445
A04         20445
A05         20445
A06         20445
A07         20445
A08         20445
bonafide    18797
Name: sample_class, dtype: int64

DEV rows = 140950
speakers = 785
A09         13702
A10         13702
A11         13702
A12         13702
A13         13702
A14         13702
A15         13702
A16         13702
bonafide    31334
Name: sample_class, dtype: int64


In [6]:
def speaker_class_table(df_part: pd.DataFrame, target_classes: list[str]) -> pd.DataFrame:
    piv = df_part.groupby(['speaker_id', 'sample_class']).size().unstack(fill_value=0)
    for c in target_classes:
        if c not in piv.columns:
            piv[c] = 0
    piv = piv[target_classes]
    meta = df_part.groupby('speaker_id')['gender'].first().rename('gender')
    piv = piv.join(meta, how='left').reset_index()
    piv['coverage_total'] = piv[target_classes].sum(axis=1)
    return piv


def split_quota(sel: pd.DataFrame, target_classes: list[str]) -> tuple[int, int]:
    tr = sel[sel['split'].eq('train')]
    te = sel[sel['split'].eq('test')]
    tr_q = int(tr[target_classes].sum().min())
    te_q = int(te[target_classes].sum().min())
    return tr_q, te_q


def split_score(train_q: int, test_q: int) -> tuple[int, int, int]:
    return (train_q + test_q, min(train_q, test_q), test_q)


def choose_full_coverage_speakers(df_part: pd.DataFrame, target_classes: list[str]) -> pd.DataFrame:
    tab = speaker_class_table(df_part, target_classes)
    eligible = tab[(tab[target_classes] > 0).all(axis=1)].copy().reset_index(drop=True)
    return eligible.sort_values(['coverage_total', 'speaker_id'], ascending=[False, True]).reset_index(drop=True)


def choose_best_split(eligible: pd.DataFrame, target_classes: list[str], seed: int):
    n_speakers = len(eligible)
    assert n_speakers >= 3, f'Need at least 3 eligible speakers, got {n_speakers}'

    train_spk = max(int(round(n_speakers * TRAIN_SPEAKER_RATIO)), 1)
    train_spk = min(train_spk, n_speakers - MIN_TEST_SPEAKERS)
    test_spk = n_speakers - train_spk
    assert test_spk >= MIN_TEST_SPEAKERS, (train_spk, test_spk)

    rng = np.random.default_rng(seed)
    best = None
    best_score = None

    speaker_ids = eligible['speaker_id'].tolist()
    for _ in range(MAX_SELECTION_TRIES):
        perm = rng.permutation(n_speakers)
        tr_idx = perm[:train_spk]
        te_idx = perm[train_spk:]
        tr = eligible.iloc[tr_idx].copy()
        te = eligible.iloc[te_idx].copy()
        tr['split'] = 'train'
        te['split'] = 'test'
        out = pd.concat([tr, te], ignore_index=True)
        train_q, test_q = split_quota(out, target_classes)
        cur = split_score(train_q, test_q)
        if best_score is None or cur > best_score:
            best_score = cur
            best = out.copy()

    # deterministic fallback by coverage rank split
    ordered = eligible.sort_values(['coverage_total', 'speaker_id'], ascending=[False, True]).reset_index(drop=True)
    tr = ordered.iloc[:train_spk].copy()
    te = ordered.iloc[train_spk:].copy()
    tr['split'] = 'train'
    te['split'] = 'test'
    fallback = pd.concat([tr, te], ignore_index=True)
    fb_q = split_quota(fallback, target_classes)
    fb_score = split_score(*fb_q)
    if best_score is None or fb_score > best_score:
        best = fallback
        best_score = fb_score

    train_q, test_q = split_quota(best, target_classes)
    return best, train_spk, test_spk, train_q, test_q


def sample_manifest(df_part: pd.DataFrame, speaker_plan: pd.DataFrame, source_partition: str, target_classes: list[str], train_q: int, test_q: int, seed: int):
    rng = np.random.default_rng(seed)
    selected_rows = []
    audit_rows = []

    for split_name, target_n in [('train', train_q), ('test', test_q)]:
        spk_set = set(speaker_plan.loc[speaker_plan['split'].eq(split_name), 'speaker_id'].astype(str).tolist())
        split_df = df_part[df_part['speaker_id'].astype(str).isin(spk_set)].copy()

        for cls in target_classes:
            pool = split_df[split_df['sample_class'].eq(cls)].copy()
            avail = len(pool)
            if avail < target_n:
                raise RuntimeError(f'Insufficient pool for {source_partition}/{cls} in {split_name}: avail={avail}, need={target_n}')
            pick_idx = rng.choice(pool.index.to_numpy(), size=target_n, replace=False)
            pick = pool.loc[pick_idx].copy()
            pick['split'] = split_name
            pick['target_class'] = cls
            selected_rows.append(pick)
            audit_rows.append({
                'source_partition': source_partition,
                'split': split_name,
                'target_class': cls,
                'target_n': int(target_n),
                'selected_n': int(len(pick)),
                'availability_n': int(avail),
            })

    manifest = pd.concat(selected_rows, ignore_index=True)
    assert manifest['utt_id'].nunique() == len(manifest), f'Duplicate utt_id selected for {source_partition}'

    manifest = manifest[[
        'source_partition', 'split', 'speaker_id', 'utt_id', 'gender', 'label', 'system_id', 'sample_class', 'codec_id', 'target_class'
    ]].sort_values(['split', 'target_class', 'speaker_id', 'utt_id']).reset_index(drop=True)

    audit_df = pd.DataFrame(audit_rows).sort_values(['split', 'target_class']).reset_index(drop=True)
    return manifest, audit_df


In [7]:
# ===== Find best split per source partition =====
partition_plans = {}

for i, (source_partition, spec) in enumerate(PARTITION_SPECS.items()):
    df_part = partition_frames[source_partition].copy().reset_index(drop=True)
    target_classes = ['bonafide'] + spec['target_systems']
    eligible = choose_full_coverage_speakers(df_part, target_classes)

    speaker_plan, train_spk, test_spk, part_train_q, part_test_q = choose_best_split(
        eligible=eligible,
        target_classes=target_classes,
        seed=SEED + 100 * (i + 1),
    )
    speaker_plan = speaker_plan.copy()
    speaker_plan['source_partition'] = source_partition

    partition_plans[source_partition] = {
        'df_part': df_part,
        'target_classes': target_classes,
        'eligible': eligible,
        'speaker_plan': speaker_plan[['source_partition', 'split', 'speaker_id', 'gender'] + target_classes + ['coverage_total']],
        'train_spk': train_spk,
        'test_spk': test_spk,
        'part_train_q': part_train_q,
        'part_test_q': part_test_q,
    }

    print(f'\n{source_partition.upper()} eligible full-coverage speakers =', len(eligible))
    print(f'{source_partition.upper()} split: train_speakers={train_spk}, test_speakers={test_spk}')
    print(f'{source_partition.upper()} max balanced quotas before global harmonization: train={part_train_q}, test={part_test_q}')
    display(partition_plans[source_partition]['speaker_plan'].head(20))

GLOBAL_TRAIN_PER_CLASS = min(v['part_train_q'] for v in partition_plans.values())
GLOBAL_TEST_PER_CLASS = min(v['part_test_q'] for v in partition_plans.values())

print('\nGLOBAL_TRAIN_PER_CLASS =', GLOBAL_TRAIN_PER_CLASS)
print('GLOBAL_TEST_PER_CLASS =', GLOBAL_TEST_PER_CLASS)
print('GLOBAL_TOTAL_PER_CLASS =', GLOBAL_TRAIN_PER_CLASS + GLOBAL_TEST_PER_CLASS)



TRAIN eligible full-coverage speakers = 400
TRAIN split: train_speakers=280, test_speakers=120
TRAIN max balanced quotas before global harmonization: train=12891, test=5906


,source_partition,split,speaker_id,gender,bonafide,A01,A02,A03,A04,A05,A06,A07,A08,coverage_total
0,train,train,T_2840,F,50,30,30,30,30,30,30,30,30,290
1,train,train,T_1077,M,50,60,60,60,60,60,60,60,60,530
2,train,train,T_0959,M,50,30,30,30,30,30,30,30,30,290
3,train,train,T_0730,M,50,60,60,60,60,60,60,60,60,530
4,train,train,T_3936,F,50,30,30,30,30,30,30,30,30,290
5,train,train,T_4830,M,50,60,60,60,60,60,60,60,60,530
6,train,train,T_3093,F,50,60,60,60,60,60,60,60,60,530
7,train,train,T_4305,M,50,60,60,60,60,60,60,60,60,530
8,train,train,T_1539,F,50,60,60,60,60,60,60,60,60,530
9,train,train,T_5003,M,50,60,60,60,60,60,60,60,60,530



DEV eligible full-coverage speakers = 392
DEV split: train_speakers=274, test_speakers=118
DEV max balanced quotas before global harmonization: train=9227, test=4475


,source_partition,split,speaker_id,gender,bonafide,A09,A10,A11,A12,A13,A14,A15,A16,coverage_total
0,dev,train,D_0151,F,50,39,39,39,39,39,39,39,39,362
1,dev,train,D_1125,F,21,16,16,16,16,16,16,16,16,149
2,dev,train,D_4874,F,50,42,42,42,42,42,42,42,42,386
3,dev,train,D_0123,M,8,19,19,19,19,19,19,19,19,160
4,dev,train,D_4721,M,50,44,44,44,44,44,44,44,44,402
5,dev,train,D_3931,F,50,52,52,52,52,52,52,52,52,466
6,dev,train,D_5080,M,50,36,36,36,36,36,36,36,36,338
7,dev,train,D_0182,F,8,17,17,17,17,17,17,17,17,144
8,dev,train,D_3206,M,7,15,15,15,15,15,15,15,15,127
9,dev,train,D_0368,F,50,39,39,39,39,39,39,39,39,362



GLOBAL_TRAIN_PER_CLASS = 9227
GLOBAL_TEST_PER_CLASS = 4475
GLOBAL_TOTAL_PER_CLASS = 13702


In [8]:
# ===== Build outputs using global common quotas =====
results = {}

for i, (source_partition, spec) in enumerate(PARTITION_SPECS.items()):
    print(f'\n=== Building {source_partition} outputs ===')
    plan = partition_plans[source_partition]
    df_part = plan['df_part']
    target_classes = plan['target_classes']
    speaker_plan = plan['speaker_plan'].copy()

    manifest, audit_df = sample_manifest(
        df_part=df_part,
        speaker_plan=speaker_plan,
        source_partition=source_partition,
        target_classes=target_classes,
        train_q=GLOBAL_TRAIN_PER_CLASS,
        test_q=GLOBAL_TEST_PER_CLASS,
        seed=SEED + 1000 * (i + 1),
    )

    assert len(set(speaker_plan[speaker_plan['split'] == 'train']['speaker_id']).intersection(
        set(speaker_plan[speaker_plan['split'] == 'test']['speaker_id'])
    )) == 0, f'speaker leakage in {source_partition}'

    expected_total = GLOBAL_TRAIN_PER_CLASS + GLOBAL_TEST_PER_CLASS
    class_totals = manifest['target_class'].value_counts().to_dict()
    for c in target_classes:
        assert int(class_totals.get(c, 0)) == expected_total, f'{source_partition} wrong total for {c}'

    split_class = manifest.groupby(['split', 'target_class']).size().unstack(fill_value=0)
    for c in target_classes:
        assert int(split_class.loc['train', c]) == GLOBAL_TRAIN_PER_CLASS
        assert int(split_class.loc['test', c]) == GLOBAL_TEST_PER_CLASS

    out_dir = OUT_BASE / source_partition
    out_dir.mkdir(parents=True, exist_ok=True)

    speakers_csv = out_dir / 'selected_speakers.csv'
    manifest_csv = out_dir / 'selected_utterances_plan.csv'
    audit_csv = out_dir / 'selection_audit.csv'
    summary_json = out_dir / 'plan_summary.json'

    speaker_plan.to_csv(speakers_csv, index=False)
    manifest.to_csv(manifest_csv, index=False)
    audit_df.to_csv(audit_csv, index=False)

    summary = {
        'source_partition': source_partition,
        'seed': int(SEED + 100 * (i + 1)),
        'protocol': str(spec['protocol_path']),
        'selection_policy': {
            'full_coverage_speakers_only': True,
            'train_speaker_ratio': TRAIN_SPEAKER_RATIO,
            'global_train_per_class': int(GLOBAL_TRAIN_PER_CLASS),
            'global_test_per_class': int(GLOBAL_TEST_PER_CLASS),
            'global_total_per_class': int(expected_total),
            'partition_max_before_global_harmonization': {
                'train_per_class': int(plan['part_train_q']),
                'test_per_class': int(plan['part_test_q']),
            },
        },
        'speaker_plan': {
            'eligible_speakers': int(len(plan['eligible'])),
            'train_speakers': int(plan['train_spk']),
            'test_speakers': int(plan['test_spk']),
            'selected_speakers': speaker_plan[['split', 'speaker_id', 'gender']].sort_values(['split', 'speaker_id']).to_dict(orient='records'),
        },
        'class_targets': {
            'classes': target_classes,
            'total_per_class': int(expected_total),
            'train_per_class': int(GLOBAL_TRAIN_PER_CLASS),
            'test_per_class': int(GLOBAL_TEST_PER_CLASS),
        },
        'counts': {
            'manifest_rows': int(len(manifest)),
            'unique_utt_id': int(manifest['utt_id'].nunique()),
            'class_totals': {k: int(v) for k, v in manifest['target_class'].value_counts().sort_index().to_dict().items()},
            'split_class_totals': {
                s: {k: int(v) for k, v in g['target_class'].value_counts().sort_index().to_dict().items()}
                for s, g in manifest.groupby('split')
            },
        },
    }
    summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    results[source_partition] = {
        'out_dir': str(out_dir),
        'eligible_speakers': int(len(plan['eligible'])),
        'train_speakers': int(plan['train_spk']),
        'test_speakers': int(plan['test_spk']),
        'train_per_class': int(GLOBAL_TRAIN_PER_CLASS),
        'test_per_class': int(GLOBAL_TEST_PER_CLASS),
        'class_totals': {k: int(v) for k, v in manifest['target_class'].value_counts().sort_index().to_dict().items()},
    }

    print('Saved ->', out_dir)
    print('harmonized quotas: train_per_class =', GLOBAL_TRAIN_PER_CLASS, '| test_per_class =', GLOBAL_TEST_PER_CLASS)
    display(speaker_plan.head(20))
    display(audit_df.head(20))
    display(manifest.head(20))

print('\nDone. Summary:')
print(json.dumps(results, indent=2))



=== Building train outputs ===
Saved -> /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs/train
harmonized quotas: train_per_class = 9227 | test_per_class = 4475


,source_partition,split,speaker_id,gender,bonafide,A01,A02,A03,A04,A05,A06,A07,A08,coverage_total
0,train,train,T_2840,F,50,30,30,30,30,30,30,30,30,290
1,train,train,T_1077,M,50,60,60,60,60,60,60,60,60,530
2,train,train,T_0959,M,50,30,30,30,30,30,30,30,30,290
3,train,train,T_0730,M,50,60,60,60,60,60,60,60,60,530
4,train,train,T_3936,F,50,30,30,30,30,30,30,30,30,290
5,train,train,T_4830,M,50,60,60,60,60,60,60,60,60,530
6,train,train,T_3093,F,50,60,60,60,60,60,60,60,60,530
7,train,train,T_4305,M,50,60,60,60,60,60,60,60,60,530
8,train,train,T_1539,F,50,60,60,60,60,60,60,60,60,530
9,train,train,T_5003,M,50,60,60,60,60,60,60,60,60,530


,source_partition,split,target_class,target_n,selected_n,availability_n
0,train,test,A01,4475,4475,6210
1,train,test,A02,4475,4475,6210
2,train,test,A03,4475,4475,6210
3,train,test,A04,4475,4475,6210
4,train,test,A05,4475,4475,6210
5,train,test,A06,4475,4475,6210
6,train,test,A07,4475,4475,6210
7,train,test,A08,4475,4475,6210
8,train,test,bonafide,4475,4475,5906
9,train,train,A01,9227,9227,14235


,source_partition,split,speaker_id,utt_id,gender,label,system_id,sample_class,codec_id,target_class
0,train,test,T_0039,T_0000001411,F,spoof,A01,A01,AC2,A01
1,train,test,T_0039,T_0000003977,F,spoof,A01,A01,AC2,A01
2,train,test,T_0039,T_0000010295,F,spoof,A01,A01,AC2,A01
3,train,test,T_0039,T_0000017554,F,spoof,A01,A01,AC2,A01
4,train,test,T_0039,T_0000021681,F,spoof,A01,A01,AC2,A01
5,train,test,T_0039,T_0000032942,F,spoof,A01,A01,AC2,A01
6,train,test,T_0039,T_0000039301,F,spoof,A01,A01,AC2,A01
7,train,test,T_0039,T_0000043732,F,spoof,A01,A01,AC2,A01
8,train,test,T_0039,T_0000057819,F,spoof,A01,A01,AC2,A01
9,train,test,T_0039,T_0000068392,F,spoof,A01,A01,AC2,A01



=== Building dev outputs ===
Saved -> /home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs/dev
harmonized quotas: train_per_class = 9227 | test_per_class = 4475


,source_partition,split,speaker_id,gender,bonafide,A09,A10,A11,A12,A13,A14,A15,A16,coverage_total
0,dev,train,D_0151,F,50,39,39,39,39,39,39,39,39,362
1,dev,train,D_1125,F,21,16,16,16,16,16,16,16,16,149
2,dev,train,D_4874,F,50,42,42,42,42,42,42,42,42,386
3,dev,train,D_0123,M,8,19,19,19,19,19,19,19,19,160
4,dev,train,D_4721,M,50,44,44,44,44,44,44,44,44,402
5,dev,train,D_3931,F,50,52,52,52,52,52,52,52,52,466
6,dev,train,D_5080,M,50,36,36,36,36,36,36,36,36,338
7,dev,train,D_0182,F,8,17,17,17,17,17,17,17,17,144
8,dev,train,D_3206,M,7,15,15,15,15,15,15,15,15,127
9,dev,train,D_0368,F,50,39,39,39,39,39,39,39,39,362


,source_partition,split,target_class,target_n,selected_n,availability_n
0,dev,test,A09,4475,4475,4475
1,dev,test,A10,4475,4475,4475
2,dev,test,A11,4475,4475,4475
3,dev,test,A12,4475,4475,4475
4,dev,test,A13,4475,4475,4475
5,dev,test,A14,4475,4475,4475
6,dev,test,A15,4475,4475,4475
7,dev,test,A16,4475,4475,4475
8,dev,test,bonafide,4475,4475,5382
9,dev,train,A09,9227,9227,9227


,source_partition,split,speaker_id,utt_id,gender,label,system_id,sample_class,codec_id,target_class
0,dev,test,D_0087,D_0000093850,F,spoof,A09,A09,AC1,A09
1,dev,test,D_0087,D_0000417082,F,spoof,A09,A09,AC1,A09
2,dev,test,D_0087,D_0000954472,F,spoof,A09,A09,AC1,A09
3,dev,test,D_0087,D_0001261996,F,spoof,A09,A09,AC1,A09
4,dev,test,D_0087,D_0001839706,F,spoof,A09,A09,AC1,A09
5,dev,test,D_0087,D_0001918855,F,spoof,A09,A09,AC1,A09
6,dev,test,D_0087,D_0002015455,F,spoof,A09,A09,AC1,A09
7,dev,test,D_0087,D_0002265082,F,spoof,A09,A09,AC1,A09
8,dev,test,D_0087,D_0002308006,F,spoof,A09,A09,AC1,A09
9,dev,test,D_0087,D_0002310946,F,spoof,A09,A09,AC1,A09



Done. Summary:
{
  "train": {
    "out_dir": "/home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs/train",
    "eligible_speakers": 400,
    "train_speakers": 280,
    "test_speakers": 120,
    "train_per_class": 9227,
    "test_per_class": 4475,
    "class_totals": {
      "A01": 13702,
      "A02": 13702,
      "A03": 13702,
      "A04": 13702,
      "A05": 13702,
      "A06": 13702,
      "A07": 13702,
      "A08": 13702,
      "bonafide": 13702
    }
  },
  "dev": {
    "out_dir": "/home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs/dev",
    "eligible_speakers": 392,
    "train_speakers": 274,
    "test_speakers": 118,
    "train_per_class": 9227,
    "test_per_class": 4475,
    "class_totals": {
      "A09": 13702,
      "A10": 13702,
      "A11": 13702,
      "A12": 13702,
      "A13": 13702,
      "A14": 13702,
      "A15": 13702,
      "A16": 13702,
      "bonafide": 13702
 